In [45]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [46]:
df_green = spark.read.parquet('../code/data/pq/green/*/*')

In [6]:
```
SELECT 
    -- Revenue grouping 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_of_records

    -- Additional calculations

FROM green
WHERE pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY 1, 2
```

SyntaxError: invalid syntax (2354800597.py, line 1)

In [47]:
rdd = df_green \
    .select('lpep_pickup_datetime','PULocationID', 'total_amount') \
    .rdd

In [48]:
rdd.take(5)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 18, 14, 54, 39), PULocationID=66, total_amount=31.86),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 21, 17, 55, 7), PULocationID=74, total_amount=12.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 24, 8, 15, 41), PULocationID=82, total_amount=11.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 29, 14, 18), PULocationID=205, total_amount=32.61),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 5, 21, 54, 45), PULocationID=65, total_amount=12.36)]

In [49]:
from datetime import datetime

start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [50]:
rows = rdd.take(10)
row = rows[0]
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 18, 14, 54, 39), PULocationID=66, total_amount=31.86)

In [51]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)


In [52]:
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)


In [53]:
from collections import namedtuple

In [54]:
RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [55]:
def unwrap(row):
    return RevenueRow(
        hour=row[0][0],
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )


In [58]:
from pyspark.sql import types

In [59]:
df_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True), 
    types.StructField('zone', types.IntegerType(), True), 
    types.StructField('revenue', types.DoubleType(), True), 
    types.StructField('count', types.IntegerType(), True)
])

In [60]:
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(df_schema)

In [62]:
df_result.show()

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-24 08:00:00|  82|             62.55|    7|
|2020-01-07 15:00:00|  82|            574.16|   38|
|2020-01-16 19:00:00|  75|1296.8899999999987|   93|
|2020-01-07 22:00:00|  53|             122.2|    4|
|2020-01-16 10:00:00| 242|            242.41|    9|
|2020-01-15 10:00:00|  33| 491.1200000000001|   27|
|2020-01-16 10:00:00|  74| 1021.219999999999|   80|
|2020-01-07 12:00:00| 141|             40.04|    2|
|2020-01-16 13:00:00|  52|122.70999999999998|   11|
|2020-01-07 16:00:00|  25| 759.3399999999996|   29|
|2020-01-31 08:00:00|  42|            586.43|   53|
|2020-01-29 13:00:00|  74| 686.4699999999998|   61|
|2020-01-29 19:00:00|  74| 909.8199999999994|   70|
|2020-01-19 17:00:00| 260|317.43000000000006|   21|
|2020-01-29 14:00:00| 257|             26.87|    1|
|2020-01-04 20:00:00| 129| 583.2700000000002|   38|
|2020-01-15 

In [63]:
df_result.write.parquet('tmp/green-revenue')

25/03/06 04:57:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [84]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd


In [86]:
duration_rdd.take(5)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 18, 14, 54, 39), PULocationID=66, DOLocationID=161, trip_distance=6.95),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 21, 17, 55, 7), PULocationID=74, DOLocationID=42, trip_distance=1.61),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 24, 8, 15, 41), PULocationID=82, DOLocationID=173, trip_distance=2.05),
 Row(VendorID=None, lpep_pickup_datetime=datetime.datetime(2020, 1, 29, 14, 18), PULocationID=205, DOLocationID=95, trip_distance=5.32),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 5, 21, 54, 45), PULocationID=65, DOLocationID=49, trip_distance=1.94)]

In [75]:
import pandas as pd

In [77]:
rows = duration_rdd.take(10)

In [79]:
pd.DataFrame(rows, columns=columns)

,VendorID,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance
0,2.0,2020-01-18 14:54:39,66,161,6.95
1,2.0,2020-01-21 17:55:07,74,42,1.61
2,2.0,2020-01-24 08:15:41,82,173,2.05
3,NaN,2020-01-29 14:18:00,205,95,5.32
4,2.0,2020-01-05 21:54:45,65,49,1.94
5,NaN,2020-01-07 05:18:00,136,107,12.49
6,2.0,2020-01-11 14:06:26,31,241,2.08
7,NaN,2020-01-22 12:42:00,155,39,2.99
8,2.0,2020-01-10 20:32:24,74,166,2.31
9,1.0,2020-01-21 16:09:22,244,142,5.80


In [87]:
def model_predict(df):
    y_pred = df.trip_distance * 5
    return y_pred

In [90]:
def apply_model_in_batch(partition):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)
    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row
        

In [93]:
duration_rdd.mapPartitions(apply_model_in_batch).toDF().show()

+-----+--------+--------------------+------------+------------+-------------+------------------+
|Index|VendorID|lpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|predicted_duration|
+-----+--------+--------------------+------------+------------+-------------+------------------+
|    0|     2.0|                  {}|          66|         161|         6.95|             34.75|
|    1|     2.0|                  {}|          74|          42|         1.61|              8.05|
|    2|     2.0|                  {}|          82|         173|         2.05|             10.25|
|    3|     NaN|                  {}|         205|          95|         5.32|              26.6|
|    4|     2.0|                  {}|          65|          49|         1.94|               9.7|
|    5|     NaN|                  {}|         136|         107|        12.49|             62.45|
|    6|     2.0|                  {}|          31|         241|         2.08|              10.4|
|    7|     NaN|              